# 変成将棋 AlphaZero 学習（NativeAOT + Batched Gumbel MCTS）
**GPU ランタイム（T4 推奨）を選択してから実行してください。**

Python.NET 不要！NativeAOT ctypes で C# ブリッジを 47 倍高速化。

In [ ]:
# ── 1. .NET 8 + NativeAOT ビルド依存ライブラリ ──────────────────
!curl -sSL https://dot.net/v1/dotnet-install.sh | bash -s -- --channel 8.0
!apt-get install -y -q clang zlib1g-dev   # NativeAOT の Linux リンカーに必要
import os
os.environ['DOTNET_ROOT'] = '/root/.dotnet'
os.environ['PATH'] = '/root/.dotnet:' + os.environ.get('PATH', '')
!dotnet --version

In [ ]:
# ── 2. Google Drive マウント ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/変則将棋-成変更将棋/変則将棋.vscode'
print('存在確認:', os.path.exists(PROJECT_DIR))

In [ ]:
# ── 3. Python パッケージ（Python.NET 不要！）────────────────────
!pip install -q tqdm
!pip install -q torch --index-url https://download.pytorch.org/whl/cu124

import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── 4. NativeAOT DLL を Linux 向けにビルド ──────────────────────
import subprocess, shutil, os

# Drive 上の Engine.Native をローカルにコピー（Drive FUSE でのビルドを避ける）
native_src = os.path.join(PROJECT_DIR, '変成将棋.Engine.Native')
engine_src = os.path.join(PROJECT_DIR, '変成将棋.Engine')
LOCAL_BUILD = '/content/shogi_build'

if os.path.exists(LOCAL_BUILD):
    shutil.rmtree(LOCAL_BUILD)
os.makedirs(LOCAL_BUILD)
shutil.copytree(native_src, f'{LOCAL_BUILD}/変成将棋.Engine.Native')
shutil.copytree(engine_src, f'{LOCAL_BUILD}/変成将棋.Engine')

# NativeAOT ビルド（linux-x64）
env_build = {**os.environ, 'DOTNET_ROOT': '/root/.dotnet',
             'PATH': '/root/.dotnet:' + os.environ.get('PATH', '')}
r = subprocess.run(
    ['/root/.dotnet/dotnet', 'publish',
     f'{LOCAL_BUILD}/変成将棋.Engine.Native',
     '-r', 'linux-x64', '-c', 'Release', '--self-contained'],
    capture_output=True, text=True, env=env_build
)
print('return code:', r.returncode)
if r.returncode != 0:
    print(r.stderr[-1000:])
else:
    publish_dir = f'{LOCAL_BUILD}/変成将棋.Engine.Native/bin/Release/net8.0/linux-x64/publish'
    libs = [f for f in os.listdir(publish_dir) if f.endswith('.so')]
    print('生成ファイル:', libs)
    LIB_PATH = os.path.join(publish_dir, libs[0])
    os.environ['SHOGI_NATIVE_LIB'] = LIB_PATH
    print('SHOGI_NATIVE_LIB =', LIB_PATH)

In [ ]:
# ── 5. NativeAOT 接続確認（Python.NET 不要）────────────────────
import sys, time, ctypes
ML_DIR = os.path.join(PROJECT_DIR, 'ml')
if ML_DIR not in sys.path:
    sys.path.insert(0, ML_DIR)

from native_game_env import NativeGameEnv
env  = NativeGameEnv()   # SHOGI_NATIVE_LIB から自動検出
sfen = env.initial_sfen()
m    = env.legal_moves(sfen)
print('初期SFEN:', sfen[:40], '...')
print('合法手数:', len(m))

# expand API の速度確認
t = time.perf_counter()
for _ in range(1000):
    env.expand(sfen, m[0])
ms = (time.perf_counter() - t) / 1000 * 1000
print(f'shogi_expand 1000回: {ms:.3f}ms/call')
print('（Python.NET は約 8ms/call → 大幅高速化）')

In [ ]:
# ── 6. GPU・MCTS 速度確認 ────────────────────────────────────────
import numpy as np
from network            import build_net
from batched_mcts_native import BatchedNativeMCTS

net, device = build_net()
tensor = env.to_tensor(sfen)

# NN バッチ推論速度
for b in [1, 8, 16]:
    batch = np.stack([tensor] * b)
    t = time.time()
    for _ in range(50):
        net.predict_batch(batch, device)
    ms = (time.time() - t) / 50 / b * 1000
    print(f'NN batch={b:2d}: {ms:.2f}ms/推論')

# MCTS 速度推定
mcts = BatchedNativeMCTS(env, net, device, n_sims=100, batch_size=8)
t = time.time()
mcts.run(sfen)
elapsed = time.time() - t
print(f'\nMCTS 100 sims (batch=8): {elapsed:.2f}秒')
print(f'400 sims 推定: {elapsed*4:.1f}秒/手')

In [ ]:
# ── 7a. TensorBoard 起動（学習中にリアルタイム表示）───────────────
# イベントファイルはローカルに書く（Drive FUSE は検出が遅いため）
RUNS_DIR = '/content/runs'
%load_ext tensorboard
%tensorboard --logdir /content/runs

In [ ]:
# ── 7. 学習開始 ──────────────────────────────────────────────────
import pathlib, glob, json, re, importlib, sys
CKPT_DIR = os.path.join(ML_DIR, 'checkpoints')

# Drive 上の最新 train.py を確実に読み込む（キャッシュを無視）
if 'train' in sys.modules:
    importlib.reload(sys.modules['train'])
from train import run

TARGET_ITERS = 50   # ← 合計何 iter まで回すか

hp = dict(
    num_sims        = 200,
    mcts_batch_size = 16,
    train_steps     = 500,
    games_per_iter  = 30,
    min_buffer      = 300,
    save_every      = 1,
    checkpoint_dir  = CKPT_DIR,
    tensorboard_dir = '/content/runs',
)

# ── これまでの進捗を表示 ──────────────────────────────────────────
log_path = pathlib.Path(CKPT_DIR) / 'log.json'
if log_path.exists():
    with open(log_path, encoding='utf-8') as f:
        prev_log = json.load(f)
    if prev_log:
        import pandas as pd
        df = pd.DataFrame(prev_log)[
            ['iter', 'policy_loss', 'value_loss', 'ownership_loss',
             'vprefix_loss', 'buffer_size', 'elapsed']]
        df = df.rename(columns={
            'iter': 'iter', 'policy_loss': 'p_loss', 'value_loss': 'v_loss',
            'ownership_loss': 'own_loss', 'vprefix_loss': 'vp_loss',
            'buffer_size': 'buf', 'elapsed': 'sec'})
        print(f'=== これまでの進捗 ({len(prev_log)} iter 完了) ===')
        print(df.to_string(index=False))
        print(f'  最終 policy_loss : {prev_log[-1]["policy_loss"]}')
        print(f'  最終 value_loss  : {prev_log[-1]["value_loss"]}')
        print()
else:
    print('log.json なし（初回学習）')

# ── 継続学習: model_final.pt → 最新の model_iter*.pt の順に探す ──
pretrained = None
start_iter = 0

final_pt = pathlib.Path(CKPT_DIR) / 'model_final.pt'
if final_pt.exists():
    pretrained = str(final_pt)
else:
    iters = sorted(glob.glob(os.path.join(CKPT_DIR, 'model_iter*.pt')))
    if iters:
        pretrained = iters[-1]
        m = re.search(r'model_iter(\d+)\.pt$', pretrained)
        if m:
            start_iter = int(m.group(1))

remaining = max(0, TARGET_ITERS - start_iter)
if remaining == 0:
    print(f'既に {TARGET_ITERS} iter 完了済み。TARGET_ITERS を増やしてください。')
else:
    print('継続学習:', pathlib.Path(pretrained).name if pretrained else 'なし（新規）')
    if start_iter > 0:
        print(f'イテレーション: {start_iter + 1}〜{TARGET_ITERS} ({remaining} 回)')
    net = run({**hp, 'num_iters': remaining},
              pretrained_path=pretrained, start_iter=start_iter)

In [ ]:
# ── 8. ONNX エクスポート ─────────────────────────────────────────
# run() が完走した場合は net 変数を使う。
# 途中で中断した場合は EXPORT_FROM に .pt ファイル名を指定して単独実行できる。
import torch, pathlib
from network import build_net

EXPORT_FROM = None   # 例: 'model_iter0020.pt'  ← 指定しなければ自動選択

ckpt_dir = pathlib.Path(CKPT_DIR)

# エクスポート元モデルの決定
if EXPORT_FROM:
    src_pt = ckpt_dir / EXPORT_FROM
else:
    # run() の戻り値 net があればそれを使い、なければ最新 .pt を探す
    src_pt = None
    try:
        _ = net   # run() 完走後は net が定義されている
        src_pt = None   # net を直接使うので .pt 不要
    except NameError:
        candidates = sorted((ckpt_dir).glob('model_iter*.pt')) + \
                     ([ckpt_dir / 'model_final.pt']
                      if (ckpt_dir / 'model_final.pt').exists() else [])
        if candidates:
            src_pt = candidates[-1]

if src_pt is not None:
    net_exp, _ = build_net()
    net_exp.load(str(src_pt), torch.device('cpu'))
    print(f'ロード: {src_pt.name}')
else:
    net_exp = net.cpu()
    print('run() の net を使用')

onnx_path = str(ckpt_dir / 'model.onnx')
dummy = torch.zeros(1, 47, 9, 9)
torch.onnx.export(
    net_exp, dummy, onnx_path,
    input_names=['input'], output_names=['policy', 'value'],
    dynamic_axes={'input': {0: 'batch_size'}},
    opset_version=17,
)
print(f'ONNX エクスポート完了 → {onnx_path}')